# sgd-vanilla-from-scratch — faded example 1: Apply the in-place SGD parameter update

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sgd-vanilla-from-scratch`. The last cell reports your progress on the `Optimizer: SGD vanilla from scratch` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: SGD vanilla from scratch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sgd-vanilla-from-scratch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sgd-vanilla-from-scratch"
DD_SUBTOPIC = "Optimizer: SGD vanilla from scratch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The core SGD update rule is `p.array -= lr * p.grad`, applied in-place so external references to p.array remain valid. The `p.array -=` syntax modifies the tensor's data directly, while `p.array = p.array - lr * p.grad` would create a new tensor object and rebind the attribute.

## Faded exercise 1

Implement `sgd_step_single(p, lr)` that updates a single MiniTensor parameter p in-place and clears its gradient.

1. Check if `p.grad is None`; if so, return immediately.
2. Apply the SGD update in-place.
3. Set `p.grad = None`.

The blank step is the in-place update expression.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(0)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step_single(p, lr):
    if p.grad is None:
        return
    p.array -= lr * p.grad
    p.grad = None

p = MiniTensor(t.tensor([4.0]), requires_grad=True)
p.grad = t.tensor([0.8])
ptr_before = p.array.data_ptr()
sgd_step_single(p, 0.1)
ptr_after = p.array.data_ptr()

print(f'New value: {p.array.item():.4f}  (expected {4.0 - 0.1 * 0.8:.4f})')
print(f'Same storage object: {ptr_before == ptr_after}')
print(f'grad is None: {p.grad is None}')


def _test():
    import torch as t

    class MiniTensor:
        def __init__(self, array, requires_grad=False):
            self.array = array
            self.requires_grad = requires_grad
            self.grad = None
            self.recipe = None

    p = MiniTensor(t.tensor([4.0]), requires_grad=True)
    p.grad = t.tensor([0.8])
    expected_val = 4.0 - 0.1 * 0.8
    sgd_step_single(p, 0.1)
    assert abs(p.array.item() - expected_val) < 1e-6, f'value: {p.array.item()} expected: {expected_val}'
    assert p.grad is None, 'grad should be None after step'
    # Test None grad path
    p2 = MiniTensor(t.tensor([9.0]), requires_grad=True)
    sgd_step_single(p2, 0.1)  # should not crash
    assert p2.array.item() == 9.0, 'frozen param should be unchanged'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step_single(p, lr):
    if p.grad is None:
        return
    p.array -= lr * p.grad
    p.grad = None

p = MiniTensor(t.tensor([4.0]), requires_grad=True)
p.grad = t.tensor([0.8])
ptr_before = p.array.data_ptr()
sgd_step_single(p, 0.1)
ptr_after = p.array.data_ptr()

print(f'New value: {p.array.item():.4f}  (expected {4.0 - 0.1 * 0.8:.4f})')
print(f'Same storage object: {ptr_before == ptr_after}')
print(f'grad is None: {p.grad is None}')
```
</details>